# Deep Learning - Assignment 2
### Training-Testing CNN with CIFAR Dataset using PyTorch
#### Group No. 5

#### Setup

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fixes the OpenMP error with Anaconda on Windows

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Use the GPU if available, otherwise the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


#### Task 1

In [2]:
# Directory where the dataset will be downloaded
DATA_PATH = "./data"

# CIFAR-10 class names
classes = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")

# Basic transform: convert to tensor and normalize each RGB channel
# (mean and std of the CIFAR-10 training set)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Download and load CIFAR-10
train_set = torchvision.datasets.CIFAR10(root=DATA_PATH, train=True,  download=True, transform=basic_transform)
test_set  = torchvision.datasets.CIFAR10(root=DATA_PATH, train=False, download=True, transform=basic_transform)

# DataLoaders
batch_size = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=0)

print("Dataset loaded successfully!")
print("Training images:", len(train_set))
print("Test images:    ", len(test_set))
print("Classes:", classes)

100%|██████████| 170M/170M [31:13<00:00, 91.0kB/s]   


Dataset loaded successfully!
Training images: 50000
Test images:     10000
Classes: ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


#### Task 2

In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        # Block 1: 3 x 32 x 32 -> 32 x 16 x 16
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        # Block 2: 32 x 16 x 16 -> 64 x 8 x 8
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # Block 3: 64 x 8 x 8 -> 128 x 4 x 4
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)

        # Max pooling halves the height and width
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()

        # Classifier
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.5)  # randomly drops neurons to reduce overfitting
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Feature extraction
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        # Classification
        x = torch.flatten(x, 1)          # (N, 128, 4, 4) -> (N, 2048)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)                  # no softmax: CrossEntropyLoss applies it
        return x

model = CNN().to(device)  # move the network to the GPU (or CPU)
print(model)

# Verify the shapes with one batch
images, labels = next(iter(train_loader))
images = images.to(device)
print("\nInput shape: ", images.shape)
print("Output shape:", model(images).shape)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=2048, out_features=256, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)

Input shape:  torch.Size([64, 3, 32, 32])
Output shape: torch.Size([64, 10])
Trainable parameters: 620362


In [4]:
def train(net, dataloader, optimizer, criterion):
    """Train the network for one epoch. Returns the average loss and accuracy."""
    net.train()  # training mode (dropout is active)
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()              # 1. clear old gradients
        outputs = net(images)              # 2. forward pass
        loss = criterion(outputs, labels)  # 3. compute the loss
        loss.backward()                    # 4. backpropagation
        optimizer.step()                   # 5. update the weights

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100 * correct / total

In [5]:
def test(net, dataloader, criterion):
    """Evaluate the network. Returns the average loss and accuracy."""
    net.eval()  # evaluation mode (dropout is turned off)
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():  # no gradients needed for testing
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, 100 * correct / total